# TTS streaming local avec pipecat et Kokoro-82M

**Module :** 03-Audio-Orchestration  
**Niveau :** Avance  
**Technologies :** pipecat, Kokoro-82M (ONNX), kokoro-onnx  
**Matériel :** CPU seul (pas de GPU requis)  

**Navigation :** [<< 03-3](03-3-Realtime-Voice-API.ipynb) | [Index](../README.md) | [Suivant >>](../04-Applications/04-1-Educational-Audio-Content.ipynb)

***

## Objectifs pedagogiques

Un assistant vocal reagit mal quand il attend la fin d'un paragraphe pour parler. Les services TTS clouds masquent cette attente : le streaming est leur affaire interne. En local, avec un modele sur CPU, la question devient visible : combien de temps s'ecoule avant le premier son (TTFA), et combien de temps total pour produire N secondes d'audio (RTF, real-time factor).

Ce carnet mesure ces deux quantites avec **pipecat**, le framework d'orchestration vocale temps reel, branche sur **Kokoro-82M**, un modele TTS compact qui tourne sur CPU via ONNX Runtime. Trois demonstrations :

1. **RTF buffer** : synthese complete d'un texte de 62 mots, mesure du facteur temps reel.
2. **TTFA streaming** : le meme texte decoupe en phrases ; mesure du temps jusqu'au premier chunk audio.
3. **Pipeline pipecat** : un mini-pipeline `TextFrame -> TTS -> collecteur` orchestre par le runner pipecat, mesure bout-en-bout.

Partie de la serie Audio (Epic #20568), slot 03-Orchestration.


In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire
# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = True                 # desactive les widgets (batch)

# Parameters
notebook_mode = "batch"
skip_widgets = True


In [2]:
# Setup environnement et imports
import os
import sys
import time
import io
import wave
from pathlib import Path

import numpy as np
import soundfile as sf

from kokoro_onnx import Kokoro

print(f"Python {sys.version.split()[0]}")
print(f"numpy {np.__version__} | soundfile {sf.__version__}")


Python 3.13.16
numpy 2.5.4 | soundfile 0.13.1


## 1. Le modele Kokoro-82M

Kokoro est un modele TTS de 82 millions de parametres (architecture StyleTTS 2 distillee), distribue en ONNX. Il phonetise le texte via espeak-ng puis synthetise par un flux ONNX Runtime : environ 310 Mo de modele, voix incluses dans un fichier binaire separe.

Le service pipecat telecharge automatiquement les fichiers la premiere fois (cache `~/.cache/pipecat/kokoro-onnx`). Ici on fait le telechargement explicitement pour mesurer sa duree une fois pour toutes.


In [3]:
# Telechargement des modeles (cache local, reutilise aux executions suivantes)
# Ecriture dans un fichier .part puis renommage : un telechargement interrompu
# ne laisse jamais un fichier partiel que le cache croirait complet.
import requests

CACHE_DIR = Path.home() / ".cache" / "pipecat" / "kokoro-onnx"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_URL = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
VOICES_URL = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
MODEL_PATH = CACHE_DIR / "kokoro-v1.0.onnx"
VOICES_PATH = CACHE_DIR / "voices-v1.0.bin"

def ensure_model_file(url: str, dest: Path, attempts: int = 4) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"deja present : {dest.name} ({dest.stat().st_size / 1e6:.1f} Mo)")
        return
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, attempts + 1):
        start = part.stat().st_size if part.exists() else 0
        headers = {"Range": f"bytes={start}-"} if start else {}
        t0 = time.perf_counter()
        try:
            resp = requests.get(url, stream=True, timeout=(10, 60), headers=headers)
            resp.raise_for_status()
            # 206 = le serveur poursuit depuis 'start' ; 200 = reprise ignoree, on repart de zero
            mode = "ab" if start and resp.status_code == 206 else "wb"
            with open(part, mode) as fh:
                for chunk in resp.iter_content(chunk_size=1 << 20):
                    fh.write(chunk)
            part.replace(dest)
            print(f"telecharge : {dest.name} ({dest.stat().st_size / 1e6:.1f} Mo) en {time.perf_counter() - t0:.1f} s")
            return
        except (requests.RequestException, OSError) as exc:
            repris = part.stat().st_size / 1e6 if part.exists() else 0
            print(f"tentative {attempt}/{attempts} echouee ({type(exc).__name__}), {repris:.1f} Mo deja sur disque")
            time.sleep(2 * attempt)
    raise ConnectionError(f"telechargement impossible apres {attempts} tentatives : {url}")

ensure_model_file(MODEL_URL, MODEL_PATH)
ensure_model_file(VOICES_URL, VOICES_PATH)

deja present : kokoro-v1.0.onnx (325.5 Mo)
deja present : voices-v1.0.bin (28.2 Mo)


In [4]:
# Instanciation du modele brut kokoro-onnx
model = Kokoro(str(MODEL_PATH), str(VOICES_PATH))
voices = sorted(model.voices)
print(f"{len(voices)} voix disponibles, 5 premieres : {voices[:5]}")


54 voix disponibles, 5 premieres : ['af_alloy', 'af_aoede', 'af_bella', 'af_heart', 'af_jessica']


### Lecture du resultat

Le modele expose ses voix par identifiant court (`af_heart` = americaine feminine, `bf_emma` = britannique, ...). Le carnet utilise `af_heart`, la voix de reference de Kokoro v1.0.


## 2. Demo 1 : RTF en mode buffer

Le mode buffer synthetise tout le texte d'un coup : `Kokoro.create(text, ...)` retourne le signal complet puis le sample rate. Le **RTF** (real-time factor) mesure le rapport duree de synthese / duree d'audio produit : RTF < 1 signifie que la synthese va plus vite que la lecture, condition necessaire pour du temps reel.

Le texte de test fait 62 mots : un paragraphe court typique d'un assistant vocal.


In [5]:
# Demo 1 : synthese buffer, mesure RTF
TEXT_BUFFER = (
    "Le streaming vocal change tout dans un assistant : au lieu d'attendre la fin "
    "d'un paragraphe entier, le modele pousse des morceaux d'audio au fil de la "
    "synthese. L'utilisateur entend la premiere syllabe bien avant la derniere, "
    "et la conversation garde son rythme naturel. Sur CPU, ce decoupage devient "
    "une decision de conception, pas un detail d'implementation."
)

t0 = time.perf_counter()
samples, sample_rate = model.create(TEXT_BUFFER, voice="af_heart", lang="en-us", speed=1.0)
synth_s = time.perf_counter() - t0
audio_s = len(samples) / sample_rate
rtf = synth_s / audio_s

n_words = len(TEXT_BUFFER.split())
print(f"mots={n_words} | synthese={synth_s:.2f} s | audio produit={audio_s:.2f} s | RTF={rtf:.2f}")
print(f"sample_rate={sample_rate} Hz | echantillons={len(samples)}")


mots=57 | synthese=16.98 s | audio produit=22.68 s | RTF=0.75
sample_rate=24000 Hz | echantillons=544256


In [6]:
# Ecoute du rendu (commite avec le carnet)
sf.write("audio/buffer_rtf.wav", samples, sample_rate)
print("audio/buffer_rtf.wav ecrit")


audio/buffer_rtf.wav ecrit


### Lecture du resultat

Le RTF mesure ici sur CPU documente la faisabilite du temps reel : en dessous de 1.0, le modele produit le son plus vite qu'on ne le lit. C'est aussi la baseline a laquelle comparer le mode streaming de la demo suivante.


## 3. Demo 2 : TTFA en mode streaming

`Kokoro.create_stream(...)` retourne un generateur asynchrone qui yield des chunks `(samples, sample_rate)`. Le **TTFA** (time to first audio) mesure le delai entre le debut de la synthese et le premier chunk : c'est le temps que l'utilisateur attend avant d'entendre quelque chose.

Le point souvent mal compris : kokoro-onnx ne streame pas phrase par phrase. Le texte est phonetise puis decoupe en **lots de 510 phonemes maximum** (`MAX_PHONEME_LENGTH` dans `config.py` de kokoro-onnx) : le lot est l'unite de streaming, et un lot n'est emis qu'une fois synthetise en entier. Un texte court tient dans un seul lot ; un texte long en produit plusieurs.

Le protocole mesure donc deux cas : d'abord `TEXT_BUFFER` (57 mots, un seul lot attendu), puis un texte long qui depasse le seuil et fait apparaitre le vrai benefice du streaming.

In [7]:
# Demo 2 : synthese streaming, mesure TTFA sur le texte court
# Le kernel Jupyter fait deja tourner sa boucle asyncio : on await la coroutine
# directement dans la cellule (run_until_complete ne peut pas nicher une boucle
# dans une boucle deja active).

async def measure_stream(text: str):
    """Mesure TTFA et synthese totale d'un stream ; accumule les chunks."""
    t0 = time.perf_counter()
    ttfa = None
    n_chunks = 0
    chunks = []
    sr_stream = None
    stream = model.create_stream(text, voice="af_heart", lang="en-us", speed=1.0)
    async for samples_i, sr_i in stream:
        if ttfa is None:
            ttfa = time.perf_counter() - t0
        n_chunks += 1
        chunks.append(samples_i)
        sr_stream = sr_i
    total_s = time.perf_counter() - t0
    samples = np.concatenate(chunks) if chunks else np.array([])
    return ttfa, n_chunks, samples, sr_stream, total_s

ttfa, n_chunks, samples_short, sr, stream_total_s = await measure_stream(TEXT_BUFFER)
stream_audio_s = len(samples_short) / sr
print(f"TEXT_BUFFER ({n_words} mots) : chunks={n_chunks} | TTFA={ttfa:.2f} s | synthese totale={stream_total_s:.2f} s | audio={stream_audio_s:.2f} s")
print(f"ecart TTFA vs synthese buffer ({synth_s:.2f} s) : {ttfa - synth_s:+.2f} s")

TEXT_BUFFER (57 mots) : chunks=1 | TTFA=22.02 s | synthese totale=22.02 s | audio=22.68 s
ecart TTFA vs synthese buffer (16.98 s) : +5.04 s


### Lecture du resultat

Le texte court (57 mots) tient dans un seul lot de phonemes : le stream produit exactement **1 chunk**, et le TTFA vaut la synthese totale (22.02 s les deux sur ce passage). Le streaming n'apporte rien ici - il ajoute meme un cout : la meme synthese en mode buffer prenait 16.98 s, soit +5.04 s pour le passage par `create_stream` (phonetisation complete avant le premier lot, iteration du generateur).

La lecon : le TTFA ne descend jamais sous le temps de synthese d'un lot. Pour entendre le premier son plus tot, il faut un texte qui depasse 510 phonemes - c'est la demo suivante.

In [8]:
# Demo 2 bis : texte long — le vrai regime du streaming
# kokoro-onnx decoupe le texte en batches d'au plus MAX_PHONEME_LENGTH = 510
# phonemes : le batch est l'unite de stream, pas la phrase. Un texte court tient
# en un seul batch (demo 2) ; un texte long en produit plusieurs, et le premier
# chunk part pendant que la suite se synthetise encore.

TEXT_LONG = TEXT_BUFFER + (
    " Un moteur de synthese decoupe le texte en fenetres de phonemes, chaque "
    "fenetre est synthetisee puis diffusee, et l'auditeur percoit le debut du "
    "message alors que la fin est encore en cours de calcul. Plus le texte est "
    "long, plus le gain est visible : le premier son ne depend plus de la duree "
    "totale du texte, mais seulement de la premiere fenetre. C'est exactement "
    "ce que mesure le time to first audio. A l'inverse, sur un texte court qui "
    "tient en une seule fenetre, le mode streaming n'apporte rien : l'unique "
    "morceau arrive a la fin de la synthese, exactement comme en mode buffer. "
    "La longueur du texte decide donc du benefice du streaming, et un assistant "
    "vocal choisit son decoupage en fonction."
)

ttfa_long, n_chunks_long, samples_long, sr_long, total_long_s = await measure_stream(TEXT_LONG)
audio_long_s = len(samples_long) / sr_long
print(f"TEXT_LONG ({len(TEXT_LONG.split())} mots) : chunks={n_chunks_long} | TTFA={ttfa_long:.2f} s | synthese totale={total_long_s:.2f} s | audio={audio_long_s:.2f} s")
print(f"premier son disponible a {100 * ttfa_long / total_long_s:.0f} % de la synthese totale")

# Ecoute du rendu streaming (flux long reconstitue depuis les chunks)
sf.write("audio/streaming_ttfa.wav", samples_long, sr_long)
print(f"audio/streaming_ttfa.wav ecrit ({audio_long_s:.2f} s d'audio)")

TEXT_LONG (181 mots) : chunks=3 | TTFA=24.74 s | synthese totale=61.52 s | audio=67.72 s
premier son disponible a 40 % de la synthese totale
audio/streaming_ttfa.wav ecrit (67.72 s d'audio)


## 4. Demo 3 : le service pipecat KokoroTTSService

pipecat orchestre les flux temps reel par un systeme de **frames** : un texte entre, le service TTS le phonetise, le synthetise, et yield des `TTSAudioRawFrame` (audio PCM int16 + sample rate + contexte). Dans un assistant complet, ces frames partent vers un transport (WebSocket, haut-parleurs) ; ici, sans transport, on consomme le generateur `run_tts` du service directement : c'est la frontiere documentee du service, celle que le transport reel consomme lui-meme.

Deux faits mesures sur pipecat 1.16.0 motivent ce choix :

- sans transport, les frames audio ne descendent pas aux processeurs situes en aval (routage `transport_destination`) ;
- le chemin `PipelineTask(observers=[...])` leve `'TaskManager' object has no attribute 'task_manager'` (defaut de la version 1.16.0, observe a l'execution).

Le service reste l'organe reel : `KokoroTTSService` vit dans `pipecat.services.kokoro.tts` (l'import racine `pipecat.services.kokoro` n'expose pas la classe dans cette version), il applique la conversion de langue, la vitesse, et la validation de la voix.

In [9]:
# Demo 3 : service pipecat, frames consommees a la frontiere run_tts
from pipecat.services.kokoro.tts import KokoroTTSService
from pipecat.frames.frames import TTSAudioRawFrame

tts_service = KokoroTTSService(
    model_path=str(MODEL_PATH),
    voices_path=str(VOICES_PATH),
    settings=KokoroTTSService.Settings(voice="af_heart", language="en", speed=1.0),
)
print(f"service pret, voix disponibles via le modele sous-jacent : {len(model.voices)}")

async def run_service_tts(text: str):
    """Consomme le generateur run_tts du service et mesure l'arrivee des frames."""
    t0 = time.perf_counter()
    ttfa = None
    chunks = []
    sample_rate = None
    context_id = tts_service.create_context_id()
    async for frame in tts_service.run_tts(text, context_id):
        if isinstance(frame, TTSAudioRawFrame) and frame.audio:
            if ttfa is None:
                ttfa = time.perf_counter() - t0
            chunks.append(frame.audio)
            sample_rate = frame.sample_rate
    total_s = time.perf_counter() - t0
    return ttfa, chunks, sample_rate, total_s

2026-11-14 21:08:02.338 | INFO     | pipecat:<module>:54 - ᓚᘏᗢ Pipecat 1.16.0 (Python 3.13.16 (tags/v3.13.16:cbc944f, Sep 30 2026, 19:17:50) [MSC v.1944 64 bit (AMD64)]) ᓚᘏᗢ


service pret, voix disponibles via le modele sous-jacent : 54


In [10]:
# Execution du service pipecat et mesure — meme texte que la demo 2
# (TEXT_BUFFER) : l'ecart de TTFA mesure alors la surcharge reelle du
# framework sur un travail identique.

svc_ttfa, svc_chunks, svc_sr, svc_total_s = await run_service_tts(TEXT_BUFFER)
svc_bytes = sum(len(c) for c in svc_chunks)
svc_audio_s = (svc_bytes / 2) / svc_sr
print(f"chunks={len(svc_chunks)} | octets={svc_bytes} | TTFA={svc_ttfa:.2f} s | total={svc_total_s:.2f} s")
print(f"audio produit : {svc_audio_s:.2f} s @ {svc_sr} Hz")
print(f"surcharge service vs moteur brut (meme texte) : {svc_ttfa - ttfa:+.2f} s")

chunks=1 | octets=1088512 | TTFA=20.44 s | total=20.44 s
audio produit : 22.68 s @ 24000 Hz
surcharge service vs moteur brut (meme texte) : -1.58 s


### Lecture du resultat

Le service rend 1 chunk sur le meme texte que la demo 2, avec un TTFA de 20.44 s - legerement **inferieur** au moteur brut (22.02 s). Une surcharge negative n'est pas une performance : elle est dans le bruit de mesure CPU, dont les ecarts run-a-run sur ce carnet atteignent quelques secondes. Le service `KokoroTTSService` delegue en effet la synthese au meme moteur ONNX ; la conversion de langue et la validation de voix ajoutent une latence non mesurable a cette echelle.

Chaque frame rendue par `run_tts` porte son sample rate et son contexte (`context_id`) : c'est l'unite que le transport pipecat (WebSocket, transport pipecat-cloud, etc.) recevrait. La frontiere service mesuree ici est honnete : hors transport, le framework n'ajoute pas de latence detectable - la latence vient du moteur.

In [11]:
# Reconstruction de l'audio du service et ecriture
svc_audio = b"".join(svc_chunks)
svc_samples = np.frombuffer(svc_audio, dtype=np.int16).astype(np.float32) / 32767.0
sf.write("audio/service_pipecat.wav", svc_samples, svc_sr)
print(f"audio/service_pipecat.wav ecrit ({len(svc_samples) / svc_sr:.2f} s d'audio @ {svc_sr} Hz)")


audio/service_pipecat.wav ecrit (22.68 s d'audio @ 24000 Hz)


## 5. Synthese : streaming local vs service cloud

### Ce que les mesures disent

Trois regimes apparaissent sur ce CPU (ONNX Runtime, execution locale) :

| Grandeur | Valeur mesuree | Lecture |
|---|---|---|
| RTF buffer (demo 1) | 0.75 - 16.98 s de synthese pour 22.68 s d'audio | < 1 : le temps reel est possible en buffer sur CPU |
| TTFA texte court (demo 2) | 22.02 s = synthese totale (1 chunk) | un texte d'un seul lot ne gagne rien au streaming |
| TTFA texte long (demo 2) | 24.74 s sur 61.52 s (3 chunks) | premier son a 40 % de la synthese : le vrai gain du streaming |
| TTFA service pipecat (demo 3) | 20.44 s (texte court) | surcharge non mesurable : le service delegue au meme moteur |

### Les compromis

- **Qualite** : Kokoro-82M reste en dessous des TTS clouds de pointe, mais la voix `af_heart` est claire et naturelle pour de l'anglais ; le francais passe par espeak-ng (`fr-fr`) avec un accent plus marque.
- **Langue** : le carnet synthetise en anglais (`en-us`) pour beneficier des meilleures voix du modele.
- **Latence vs longueur** : le lot de 510 phonemes est l'unite de streaming ; un assistant vocal qui repond en phrases courtes reste en regime texte court (TTFA = synthese du lot), et le benefice du streaming n'apparait que sur les reponses longues.
- **Deploiement** : tout tourne en local (CPU, ONNX Runtime), sans cle API ni cout par requete ; le compromis cloud reste la qualite vocale maximale.

### Pour aller plus loin

- `pipecat.audio` : VAD, resampling et filtres pour chainer micro -> STT -> LLM -> TTS.
- Les services TTS alternatifs de pipecat (`openai`, `xtts`, `pocket_tts`) s'interchangent avec `KokoroTTSService` dans le meme pipeline.
- Le carnet 03-2 montre l'orchestration STT -> LLM -> TTS classique ; le carnet 03-3 montre l'alternative temps reel cloud (OpenAI Realtime API).

In [12]:
# Recapitulatif des mesures du carnet
print("Mesures du carnet 03-4 :")
print(f"  RTF buffer, {n_words} mots             : {rtf:.2f} (synthese {synth_s:.2f} s, audio {audio_s:.2f} s)")
print(f"  TTFA texte court ({n_chunks} chunk)    : {ttfa:.2f} s, synthese totale {stream_total_s:.2f} s")
print(f"  TTFA texte long ({n_chunks_long} chunks) : {ttfa_long:.2f} s sur {total_long_s:.2f} s de synthese")
print(f"  TTFA service pipecat (texte court)     : {svc_ttfa:.2f} s (chunks: {len(svc_chunks)})")

Mesures du carnet 03-4 :
  RTF buffer, 57 mots             : 0.75 (synthese 16.98 s, audio 22.68 s)
  TTFA texte court (1 chunk)    : 22.02 s, synthese totale 22.02 s
  TTFA texte long (3 chunks) : 24.74 s sur 61.52 s de synthese
  TTFA service pipecat (texte court)     : 20.44 s (chunks: 1)


***

**Navigation :** [<< 03-3](03-3-Realtime-Voice-API.ipynb) | [Index](../README.md) | [Suivant >>](../04-Applications/04-1-Educational-Audio-Content.ipynb)